# Week 19 · Notebook 01: Gemini Multimodal OCR

# Requirements: pip install google-genai pillow numpy pandas

# ⚠️ REQUIRES: GOOGLE_API_KEY

> 💰 COST WARNING: set a budget alert before running, Gemini vision bills per image + per token, and the free tier has a daily request quota.


## What you build

Render synthetic bills of lading to PNG locally with Pillow, then let Gemini vision extract the fields to structured JSON and score per-field accuracy against ground truth, ending with a cost-per-document estimate. The local (image + accuracy + cost) parts run without any key; only the Gemini call needs `GOOGLE_API_KEY`.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root (template)
# Robust fallback: walk up until we find zoro/data.py, in case Jupyter started elsewhere.
_root = pathlib.Path.cwd()
for _p in [_root, *_root.parents]:
    if (_p / "zoro" / "data.py").exists():
        sys.path.insert(0, str(_p))
        break

import os
import json
import numpy as np
import pandas as pd
from zoro import data

SEED = 19
rng = np.random.default_rng(SEED)
print("zoro.data ready; seed =", SEED)


## Step 1: Render BoL text onto PNGs locally

The OCR ground truth already lives in `data.bol_samples()`. We render each text block to an image so Gemini exercises *vision*, not just reading text from the prompt. This keeps the pipeline end-to-end (image → JSON) and fully local before the API call.


In [ ]:
from PIL import Image, ImageDraw, ImageFont

def render_bol_png(text, path, font_size=16, width=760):
    lines = text.strip().splitlines()
    line_h = int(font_size * 1.6)
    height = line_h * len(lines) + 60
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("DejaVuSansMono.ttf", font_size)
    except Exception:
        try:
            font = ImageFont.load_default(size=font_size)
        except TypeError:
            font = ImageFont.load_default()
    y = 20
    for line in lines:
        draw.text((20, y), line, fill="black", font=font)
        y += line_h
    img.save(path)
    return path

IMG_DIR = pathlib.Path("bol_images")
IMG_DIR.mkdir(exist_ok=True)
bols = data.bol_samples(n=6, seed=5)
img_paths = []
for b in bols:
    p = render_bol_png(b["text"], IMG_DIR / f"{b['bol_id']}.png")
    img_paths.append((b, p))
print("Rendered", len(img_paths), "BoL images into", IMG_DIR)


## Step 2: Gemini vision → structured JSON

Gemini accepts image + instruction in one call and can return JSON. We request `response_mime_type: application/json` and parse `resp.text`. A `response_schema` (structured output) is the stricter production option, see the Vertex module README for that shape.


In [ ]:
EXTRACT_PROMPT_OCR = (
    "Extract the bill-of-lading fields from this image. "
    "Return ONLY valid JSON with exactly these keys: "
    "shipper, consignee, port_of_loading, port_of_discharge, commodity, "
    "quantity, gross_weight_kg, declared_value_usd, freight_terms, date_of_issue. "
    "quantity and gross_weight_kg are integers; declared_value_usd is a number."
)

api_key = os.environ.get("GOOGLE_API_KEY")
GEMINI_READY = bool(api_key)
if not GEMINI_READY:
    print("⚠️ GOOGLE_API_KEY not found, running in DRY-RUN mode.")
    print("Get a key at https://aistudio.google.com then:")
    print(" export GOOGLE_API_KEY='<your key>'")

MODEL = "gemini-2.5-flash"

def gemini_extract(image_bytes):
    from google import genai
    client = genai.Client(api_key=api_key)
    resp = client.models.generate_content(
        model=MODEL,
        contents=[
            EXTRACT_PROMPT_OCR,
            {"inline_data": {"mime_type": "image/png", "data": image_bytes}},
        ],
        config={"response_mime_type": "application/json"},
    )
    return resp.text


## Step 3: Run extraction over N documents

Loop over the rendered images, call Gemini, and parse JSON. Keep the cloud run small (4 images) until you confirm the daily quota and cost.


In [ ]:
def normalize_string(s):
    return str(s).strip().lower()

def field_matches(pred, truth, key):
    if pred is None or truth is None:
        return False
    if key in ("quantity", "gross_weight_kg"):
        try:
            return int(float(pred)) == int(float(truth))
        except (TypeError, ValueError):
            return False
    if key == "declared_value_usd":
        try:
            return abs(float(pred) - float(truth)) < 0.01
        except (TypeError, ValueError):
            return False
    return normalize_string(pred) == normalize_string(truth)

FIELDS = ["shipper", "consignee", "port_of_loading", "port_of_discharge",
          "commodity", "quantity", "gross_weight_kg", "declared_value_usd",
          "freight_terms", "date_of_issue"]

predictions = []
usage = {"images": 0, "in_chars": 0, "out_chars": 0}

if GEMINI_READY:
    for b, p in img_paths[:4]:  # raise to 6 when confident about quota
        img_bytes = p.read_bytes()
        raw = ""
        try:
            raw = gemini_extract(img_bytes).strip()
            raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            pred = json.loads(raw)
        except Exception as e:  # noqa: BLE001
            pred = None
            print("  failed on", b["bol_id"], "->", type(e).__name__)
        predictions.append((b, pred))
        usage["images"] += 1
        usage["in_chars"] += len(EXTRACT_PROMPT_OCR)
        usage["out_chars"] += len(raw)
else:
    predictions = [(b, None) for b, _ in img_paths[:4]]
    print("DRY-RUN: skipped Gemini extraction (set GOOGLE_API_KEY).")

print("Collected predictions for", len(predictions), "documents.")


In [ ]:
per_field = {f: {"correct": 0, "total": 0} for f in FIELDS}
for b, pred in predictions:
    truth = b["fields"]
    for f in FIELDS:
        per_field[f]["total"] += 1
        if pred and field_matches(pred.get(f), truth.get(f), f):
            per_field[f]["correct"] += 1

rows = []
for f in FIELDS:
    c, t = per_field[f]["correct"], per_field[f]["total"]
    rows.append({"field": f, "correct": c, "total": t, "accuracy": (c / t) if t else 0.0})

acc_df = pd.DataFrame(rows)
overall_acc = acc_df["correct"].sum() / acc_df["total"].sum() if acc_df["total"].sum() else 0.0
print(acc_df.to_string(index=False))
print(f"\nOVERALL_FIELD_ACCURACY: {overall_acc:.3f}")


## Step 4: Cost per document

Gemini bills per image and per token (input/output, with context-caching discounts we ignore here). We estimate tokens from characters (≈ 4 chars/token) with placeholder prices you must verify against the live Gemini/Vertex pricing pages.


In [ ]:
# Placeholder prices, VERIFY against live Gemini / Vertex pricing.
PRICE_PER_IMAGE = 0.000315 # USD per image
PRICE_PER_1M_IN = 0.30 # USD per 1M input tokens
PRICE_PER_1M_OUT = 2.50 # USD per 1M output tokens

n_docs = max(usage["images"], 1)
in_tokens = usage["in_chars"] // 4
out_tokens = usage["out_chars"] // 4
total_cost = (usage["images"] * PRICE_PER_IMAGE
              + in_tokens * PRICE_PER_1M_IN / 1_000_000
              + out_tokens * PRICE_PER_1M_OUT / 1_000_000)
cost_per_doc = total_cost / n_docs

print(f"Documents: {usage['images']} · in tokens: {in_tokens:,} · out tokens: {out_tokens:,}")
print(f"Total estimated cost: ${total_cost:.6f} USD")
print(f"COST_PER_DOCUMENT_USD: {cost_per_doc:.6f}")
